In [247]:
import os
import pandas as pd
import numpy as np
import psycopg2
import warnings

In [248]:
warnings.filterwarnings('ignore')
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [249]:
def read_url(link):
    """ Creates a pandas DataFrame from data online
    - Parameters:
        - link: link to the zipped data
    - Returns:
    """
    import io
    import requests
    import pandas as pd

    # Define URL and extract information
    response = requests.get(link)
    content = response.content
    # Convert into a Pandas DataFrame
    df = pd.read_csv(io.BytesIO(content), sep=',', compression='gzip')

    return df

In [250]:
# listings = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/listings.csv.gz')
listings = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2026-03-30/data/listings.csv.gz')

In [251]:
listings[['id', 'neighborhood_overview']]

,id,neighborhood_overview
0,35797,NaN
1,70644,NaN
2,245507,NaN
3,247543,NaN
4,261897,NaN
...,...,...
22765,1649667345161605554,NaN
22766,1649679876711060058,NaN
22767,1649695882994593811,NaN
22768,1649701281434257924,NaN


In [252]:
column_dates = ['last_scraped', 'host_since', 'price_quote_checkin_date',
                'price_quote_checkout_date', 'calendar_updated', 'calendar_last_scraped',
                'first_review', 'last_review']

for col in column_dates:
    if col in listings.columns.to_list():
        listings[col] = pd.to_datetime(listings[col], errors='coerce')
        listings[col] = listings[col].dt.date
        listings[col] = listings[col].where(listings[col].notna(), None)
    
listings = listings.astype(object)
listings = listings.where(pd.notnull(listings), None)

In [253]:
if 'price' in listings.columns.to_list():
    listings['price'] = listings['price'].replace('[\$,]', '', regex=True).astype(float)
if 'host_acceptance_rate' in listings.columns.to_list():
    listings['host_acceptance_rate'] = listings['host_acceptance_rate'].replace('[%,]', '', regex=True).astype(float)
if 'host_response_rate' in listings.columns.to_list():
    listings['host_response_rate'] = listings['host_response_rate'].replace('[%,]', '', regex=True).astype(float)
if 'price_quote_total_price' in listings.columns.to_list():
    listings['price_quote_total_price'] = listings['price_quote_total_price'].replace('[\$,]', '', regex=True).astype(float)
if 'price_quote_price_per_night' in listings.columns.to_list():
    listings['price_quote_price_per_night'] = listings['price_quote_price_per_night'].replace('[\$,]', '', regex=True).astype(float)


In [254]:
def parse_bathrooms(text) -> float:
    import regex as re
    if pd.isna(text):
        return 0
    elif "Half-bath" in text:
        return 0.5
    elif "Private half-bath" in text:
        return 0.5
    elif "Shared half-bath" in text:
        return 0.5
    match = re.search(r"(\d+(?:\.\d+)?)", text)  
    if match:
        return float(match.group(1))
    return 0 
    
listings["bathrooms"] = (
    listings["bathrooms"]
    .fillna(
        listings["bathrooms_text"].apply(parse_bathrooms)
    )
)
listings['bathrooms_text'] = listings['bathrooms_text'].fillna(
    listings["bathrooms"].astype(str) + " baths")
listings['bedrooms'] = listings['bedrooms'].fillna(1)
listings['beds'] = listings['beds'].fillna(listings['accommodates'] // 2)
listings['minimum_nights'] = listings['minimum_nights'].fillna(1)
listings['maximum_nights'] = listings['maximum_nights'].fillna(365)
listings["price"] = listings["price"].fillna(
    listings.groupby(
        ["neighbourhood_cleansed", "property_type", "room_type"]
    )["price"]
    .transform("mean")
)

listings["price"] = listings["price"].fillna(
    listings.groupby(
        ["neighbourhood_cleansed"]
    )["price"]
    .transform("mean")
)

In [255]:
listings['review_scores_accuracy'] = listings['review_scores_accuracy'].fillna(-1)
listings['review_scores_communication'] = listings['review_scores_communication'].fillna(-1)
listings['review_scores_cleanliness'] = listings['review_scores_cleanliness'].fillna(-1)
listings['review_scores_location'] = listings['review_scores_location'].fillna(-1)
listings['review_scores_value'] = listings['review_scores_value'].fillna(-1)
listings['review_scores_rating'] = listings['review_scores_rating'].fillna(-1)
listings['reviews_per_month'] = listings['reviews_per_month'].fillna(-1)
listings['instant_bookable'] = listings['instant_bookable'].fillna(False)

listings['description'] = listings['description'].str.replace('<br />', '')
listings['neighborhood_overview'] = listings['neighborhood_overview'].str.replace('<br />', '')

In [256]:
# Create connection to the database and initialize it
def create_db_connection() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5433"),
        dbname=os.getenv("DB_NAME", "smartbnb"),
        user=os.getenv("DB_USER", "admin"),
        password=os.getenv("DB_PASSWORD", "admin")
    )
    return conn

def drop_connection(conn):
    conn.close()

In [257]:
conn = create_db_connection()
columns = listings.columns.to_list()
columns_names = ", ".join(columns)
placeholders = ", ".join(["%s"] * len(columns))

update_clause = ", ".join(
    [
        f"{col} = COALESCE(EXCLUDED.{col}, listings.{col})"
        for col in columns
        if col != "id"
    ]
)

query = f"""
    INSERT INTO listings ({columns_names}) 
    VALUES ({placeholders})
    ON CONFLICT (id) DO UPDATE SET {update_clause}
    """
    # ON CONFLICT (id) DO NOTHING

with conn.cursor() as cur:
    for _, row in listings.iterrows():
        values = [
            None if pd.isna(value) else value 
            for value in row
        ]
        cur.execute(query, values)

conn.commit()
#drop_connection()